In [1]:
import sys

import pandas as pd
import nibabel as nib
import numpy as np
import scipy
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, losses
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard, ReduceLROnPlateau

import os
import datetime

from pathlib import Path

from tensorflow.keras.callbacks import CSVLogger

import random
from tensorflow.keras.optimizers import Adam



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Images.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("")  # Extract to a directory

In [2]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.18.0
GPU available: []


In [3]:
def generate_visit_pairs(path_to_csv):
    df = pd.read_csv(path_to_csv, delimiter=";")

    df['CDR'] = df['CDR'].astype(str).str.replace(',', '.')
    df['CDR'] = pd.to_numeric(df['CDR'], errors='coerce')

    # Ensure consistent sorting
    df = df.sort_values(by=['Subject ID', 'Visit'])

    pairs = []

    for id, group in df.groupby('Subject ID'):
        visits = group.sort_values(by='Visit').reset_index(drop=True)

        for i in range(len(visits) - 1):
            v0 = visits.iloc[i]
            v1 = visits.iloc[i + 1]

            cdr0 = v0['CDR']
            cdr1 = v1['CDR']

            # Create progression label
            # CDR rose
            if cdr1 > cdr0:
                label = 2
            # CDR stable
            elif cdr1 == cdr0:
                label = 1
            # CDR reverted
            elif cdr1 < cdr0:
                label = 0

            pairs.append((v0['MRI ID'], v1['MRI ID'], label))

    return pairs

def resize_and_load_img(filename: str):
    img = nib.load("Images/" + filename + "/RAW/mpr-1.nifti.hdr")
    img_data = img.get_fdata()

    img_data = ((img_data - np.mean(img_data)) / np.std(img_data)).astype(np.float32)

    img_data = img_data[..., 0]

    zoom_factors = [t / c for t, c in zip((128, 128, 128, 1), img_data.shape)]
    volume = scipy.ndimage.zoom(img_data, zoom_factors, order=1)

    volume = volume[..., np.newaxis]

    return volume

pairs = generate_visit_pairs("/content/drive/MyDrive/oasis_longitudinal_demographics.csv") # LaterDO: files from cmd


In [4]:
def convnext_block(x, filters, kernel_size=7):
    # Lightweight ConvNeXt-style block for 3D
    shortcut = x
    x = layers.LayerNormalization()(x)
    x = layers.Conv3D(filters, kernel_size, padding='same')(x)  # Depthwise
    x = layers.LayerNormalization()(x)
    x = layers.Conv3D(filters, 1, padding='same')(x)  # Pointwise

    shortcut = layers.Conv3D(filters=x.shape[-1], kernel_size=1, padding='same')(shortcut)
    x = layers.Add()([shortcut, x])
    return x

def feature_extractor(input_shape):
    inputs = Input(input_shape)
    x = layers.Conv3D(16, 3, padding='same', activation='gelu')(inputs)
    x = convnext_block(x, 16)
    x = layers.MaxPooling3D(2)(x)
    x = convnext_block(x, 32)
    x = layers.MaxPooling3D(2)(x)
    x = convnext_block(x, 64)
    x = layers.GlobalAveragePooling3D()(x)
    model = Model(inputs, x, name="feature_extractor")
    return model

def build_model():
    input_1 = Input((128, 128, 128, 1))
    input_2 = Input((128, 128, 128, 1))

    extractor = feature_extractor((128, 128, 128, 1))

    emb_1 = extractor(input_1)
    emb_2 = extractor(input_2)

    merged = layers.Concatenate()([emb_1, emb_2])
    x = layers.Dense(128, activation='gelu')(merged)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='gelu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(3, activation='softmax')(x)

    model = Model([input_1, input_2], output)
    return model

model = build_model()
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_extractor   │ (None, 64)        │    975,216 │ input_layer[0][0… │
│ (Functional)        │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ feature_extracto… │
│ (Concatenate)       │                   │            │ feature_extracto… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 3)         │        195 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,000,179 (3.82 MB)

 Trainable params: 1,000,179 (3.82 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
batch_size = 1

def batch_generator(pairs, batch_size=2):
    while True:
        x1_batch = []
        x2_batch = []
        y_batch = []

        for scan1, scan2, label in pairs:
            img1 = resize_and_load_img(scan1)
            img2 = resize_and_load_img(scan2)

            x1 = np.array(img1).astype(np.float32)
            x2 = np.array(img2).astype(np.float32)
            y = np.array(label).astype(np.int32)  # Not wrapped in list

            x1_batch.append(x1)
            x2_batch.append(x2)
            y_batch.append(y)

            if len(x1_batch) == batch_size:
                # Stack into batch arrays
                yield ((np.stack(x1_batch), np.stack(x2_batch)),
                       np.array(y_batch))
                x1_batch, x2_batch, y_batch = [], [], []

pairs_shuffled = pairs[:]
random.shuffle(pairs_shuffled)

pairs_shuffled = np.array(pairs_shuffled)

val_offset = int(pairs_shuffled.shape[0]*0.7)
test_offset = int(pairs_shuffled.shape[0]*0.9)

x_train = pairs_shuffled[:val_offset]
x_val = pairs_shuffled[val_offset:test_offset]

train_gen = batch_generator(x_train, batch_size=batch_size)
val_gen = batch_generator(x_val, batch_size=batch_size)


In [ ]:
csv_logger = CSVLogger('training_log.csv', append=True)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.000001)

training_data_size = len(x_train)
validation_data_size = len(x_val)

history = model.fit(train_gen,
    steps_per_epoch=training_data_size//batch_size,
    validation_data=val_gen,
    validation_steps=validation_data_size//batch_size,
    epochs=50,
    verbose=1,
    callbacks=[early_stop, csv_logger, lr_scheduler]
)

Epoch 1/50
  2/156 ━━━━━━━━━━━━━━━━━━━━ 20:12:09 472s/step - accuracy: 1.0000 - loss: 0.5387

In [ ]:
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.plot(history.history['accuracy'], label='accuracy')
plt.xlabel("Epoch")
plt.legend()
plt.grid(True)
plt.show()